In [1]:
"""
Script 1: Zonal Statistics & Spearman Correlations
====================================================
WORKFLOW
--------
1. Load H3 hexagonal grid.
2. Count mocked war events (ACLED, VIINA, combined) per hex.
3. Compute per-hex z-score statistics from the debiased RF raster:
       z_median_damage   = median of z > 0 pixels  (0 if none)
       z_median_rewild   = median of z < 0 pixels  (0 if none)
       z_max_damage      = max z across all pixels
       z_max_rewild      = min z across all pixels
4. Count confident-change pixels from the confidence raster:
       conf_damage_px    = pixels with score 1..3
       conf_rewild_px    = pixels with score -3..-1
5. Assign majority ESA land cover per hex.
6. Export hex-level dataset (CSV + GPKG).
7. Compute Spearman correlations on THREE subsets:

       all_valid  -- every hex with raster coverage
                     (z_count > 0 AND conf_valid_px > 0).
                     Events and change can both be zero.
                     All 6 metrics computed.

       z_active   -- raster coverage AND events > 0.
                     Only z-score metrics (4).
                     Every hex with raster data has z-scores,
                     so only the event filter is needed.

       px_active  -- raster coverage AND events > 0
                     AND at least one confident change pixel
                     (conf_damage_px > 0 OR conf_rewild_px > 0).
                     Only pixel-count metrics (2).
                     This subset is always <= z_active.

   Land-cover stratification requires n >= 30 hexes.
8. Export correlations CSV.
"""

import warnings
warnings.filterwarnings('ignore')

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.features import rasterize
from scipy.stats import spearmanr
from pathlib import Path
from collections import Counter

# ====================================================================
# CONFIGURATION
# ====================================================================

WAR_EVENTS_PATH = r"C:\Users\karasoo1\Downloads\correlation\war_events4326_randomised.gpkg"
RF_PATH         = r"C:\Users\karasoo1\Downloads\correlation\Debiased_RF_Main_v16.tif"
CONF_PATH       = r"C:\Users\karasoo1\Downloads\correlation\Debiased_RF_Confidence_v16.tif"
HEX_PATH        = r"C:\Users\karasoo1\Downloads\correlation\h3_hexagons_res6.gpkg"
ESA_PATH        = r"C:\Users\karasoo1\Downloads\correlation\ESA_WorldCover_2021_Ukraine_AOI.tif"

OUT_DIR = r"C:\Users\karasoo1\Downloads\hex_analysis_mocked"

ZSCORE_BAND = 7
Z_VALID_MIN = -15.0
Z_VALID_MAX =  15.0
Z_NODATA    = -9999.0

MIN_N = 30

ESA_LEGEND = {
    10: 'Tree cover', 20: 'Shrubland', 30: 'Grassland', 40: 'Cropland',
    50: 'Built-up', 60: 'Bare / sparse', 80: 'Water bodies',
    90: 'Herbaceous wetland',
}
VALID_LC_CODES = set(ESA_LEGEND.keys())


# ====================================================================
# RASTER I/O HELPERS
# ====================================================================

def _hex_window(src, geom, transform):
    minx, miny, maxx, maxy = geom.bounds
    c0 = max(0, int(np.floor((minx - transform.c) / transform.a)))
    r0 = max(0, int(np.floor((maxy - transform.f) / transform.e)))
    c1 = min(src.width,  int(np.ceil((maxx - transform.c) / transform.a)))
    r1 = min(src.height, int(np.ceil((miny - transform.f) / transform.e)))
    if r1 <= r0 or c1 <= c0:
        return None, None
    win = rasterio.windows.Window(c0, r0, c1 - c0, r1 - r0)
    return win, src.window_transform(win)


def _masked_pixels(src, geom, transform, nodata):
    win, wt = _hex_window(src, geom, transform)
    if win is None:
        return None
    arr = src.read(1, window=win).astype(np.float32)
    mask = rasterize([(geom, 1)], out_shape=arr.shape,
                     transform=wt, fill=0, dtype='uint8')
    inside = arr[mask == 1]
    if nodata is not None:
        inside = inside[inside != nodata]
    inside = inside[np.isfinite(inside)]
    return inside if len(inside) > 0 else None


def extract_band_tif(raster_path, band_idx, tmp_path):
    with rasterio.open(raster_path) as src:
        arr = src.read(band_idx).astype(np.float32)
        nd = src.nodata
        profile = src.profile.copy()
    invalid = np.isnan(arr)
    if nd is not None:
        invalid |= (arr == nd)
    invalid |= (arr < Z_VALID_MIN) | (arr > Z_VALID_MAX)
    arr[invalid] = Z_NODATA
    profile.update(count=1, dtype='float32', nodata=Z_NODATA,
                   compress='deflate', tiled=True,
                   blockxsize=256, blockysize=256)
    with rasterio.open(tmp_path, 'w', **profile) as dst:
        dst.write(arr, 1)


# ====================================================================
# ZONAL STATISTICS
# ====================================================================

def zscore_zonal(hexgdf, zscore_tif):
    keys = ['z_median_damage', 'z_median_rewild',
            'z_max_damage', 'z_max_rewild', 'z_count']
    out = {k: [] for k in keys}
    nan_row = {k: np.nan for k in keys}
    nan_row['z_count'] = 0

    with rasterio.open(zscore_tif) as src:
        crs, nd, transform = src.crs, src.nodata, src.transform
        hex_r = hexgdf.to_crs(crs) if hexgdf.crs != crs else hexgdf
        total = len(hex_r)

        for ci, (_, row) in enumerate(hex_r.iterrows()):
            if (ci + 1) % 500 == 0:
                print(f"      {ci + 1}/{total}")
            geom = row.geometry
            if geom is None or geom.is_empty:
                for k in keys: out[k].append(nan_row[k])
                continue
            try:
                px = _masked_pixels(src, geom, transform, nd)
                if px is None:
                    for k in keys: out[k].append(nan_row[k])
                    continue
                out['z_count'].append(len(px))
                out['z_max_damage'].append(float(np.max(px)))
                out['z_max_rewild'].append(float(np.min(px)))
                pos = px[px > 0]
                neg = px[px < 0]
                out['z_median_damage'].append(
                    float(np.median(pos)) if len(pos) > 0 else 0.0)
                out['z_median_rewild'].append(
                    float(np.median(neg)) if len(neg) > 0 else 0.0)
            except Exception:
                for k in keys: out[k].append(nan_row[k])
    return out


def confidence_zonal(hexgdf, conf_path):
    dmg, rew, val = [], [], []
    with rasterio.open(conf_path) as src:
        crs, nd, transform = src.crs, src.nodata, src.transform
        hex_r = hexgdf.to_crs(crs) if hexgdf.crs != crs else hexgdf
        total = len(hex_r)
        for ci, (_, row) in enumerate(hex_r.iterrows()):
            if (ci + 1) % 500 == 0:
                print(f"      {ci + 1}/{total}")
            geom = row.geometry
            if geom is None or geom.is_empty:
                dmg.append(0); rew.append(0); val.append(0)
                continue
            try:
                px = _masked_pixels(src, geom, transform, nd)
                if px is None:
                    dmg.append(0); rew.append(0); val.append(0)
                    continue
                val.append(len(px))
                dmg.append(int(np.sum((px >= 1) & (px <= 3))))
                rew.append(int(np.sum((px <= -1) & (px >= -3))))
            except Exception:
                dmg.append(0); rew.append(0); val.append(0)
    return dmg, rew, val


def majority_landcover(hexgdf, esa_path):
    results = []
    with rasterio.open(esa_path) as src:
        crs, nd, transform = src.crs, src.nodata, src.transform
        hex_r = hexgdf.to_crs(crs) if hexgdf.crs != crs else hexgdf
        total = len(hex_r)
        for ci, (_, row) in enumerate(hex_r.iterrows()):
            if (ci + 1) % 500 == 0:
                print(f"      {ci + 1}/{total}")
            geom = row.geometry
            if geom is None or geom.is_empty:
                results.append(np.nan); continue
            try:
                win, wt = _hex_window(src, geom, transform)
                if win is None:
                    results.append(np.nan); continue
                arr = src.read(1, window=win)
                mask = rasterize([(geom, 1)], out_shape=arr.shape,
                                 transform=wt, fill=0, dtype='uint8')
                inside = arr[mask == 1]
                inside = inside[inside != 0]
                if nd is not None and nd != 0:
                    inside = inside[inside != nd]
                inside = inside[np.isin(inside, list(VALID_LC_CODES))]
                if len(inside) == 0:
                    results.append(np.nan)
                else:
                    results.append(int(Counter(inside).most_common(1)[0][0]))
            except Exception:
                results.append(np.nan)
    return results


# ====================================================================
# SUBSET DEFINITIONS
# ====================================================================

def make_subsets(hexgdf, ev_col):
    """
    Three subsets, each paired with the metrics it applies to.

    all_valid  : raster coverage.  Events/change can be zero.  -> all 6 metrics
    z_active   : raster + events > 0.                          -> 4 z-score metrics
    px_active  : raster + events > 0 + change pixels detected. -> 2 pixel metrics

    n(all_valid) >= n(z_active) >= n(px_active)
    """
    has_raster = (hexgdf['z_count'] > 0) & (hexgdf['conf_valid_px'] > 0)
    has_events = hexgdf[ev_col] > 0
    has_change = (hexgdf['conf_damage_px'] > 0) | (hexgdf['conf_rewild_px'] > 0)

    return {
        'all_valid':  has_raster,
        'z_active':   has_raster & has_events,
        'px_active':  has_raster & has_events & has_change,
    }


# ====================================================================
# METRIC DEFINITIONS
# ====================================================================

Z_METRICS = [
    ('Med damage',     'z_median_damage',   'intensity'),
    ('Med warwilding', 'z_median_rewild',   'intensity'),
    ('Max damage',     'z_max_damage',       'intensity'),
    ('Max warwilding', 'z_max_rewild',       'intensity'),
]

PX_METRICS = [
    ('Damage px',      'conf_damage_px',     'scope'),
    ('Warwilding px',  'conf_rewild_px',     'scope'),
]

# Which metrics belong to which subset
SUBSET_METRICS = {
    'all_valid':  Z_METRICS + PX_METRICS,   # all 6
    'z_active':   Z_METRICS,                 # 4 z-score only
    'px_active':  PX_METRICS,                # 2 pixel only
}


# ====================================================================
# CORRELATIONS
# ====================================================================

def compute_correlations(hexgdf, event_cols):
    rows = []

    for ev_label, ev_col in event_cols.items():
        subsets = make_subsets(hexgdf, ev_col)

        for subset_name, shared_mask in subsets.items():
            metrics = SUBSET_METRICS[subset_name]

            # All land cover
            sub = hexgdf[shared_mask]
            if len(sub) >= MIN_N:
                for m_label, m_col, m_group in metrics:
                    rho, pval = spearmanr(sub[ev_col].values,
                                          sub[m_col].values)
                    rows.append({
                        'subset': subset_name,
                        'event_source': ev_label,
                        'metric': m_label,
                        'group': m_group,
                        'land_cover': 'All',
                        'lc_code': -1,
                        'n': len(sub),
                        'spearman_rho': rho,
                        'p_value': pval,
                    })

            # Per land cover
            for lc_code, lc_name in ESA_LEGEND.items():
                lc_sub = hexgdf[shared_mask & (hexgdf['lc_code'] == lc_code)]
                if len(lc_sub) < MIN_N:
                    continue
                for m_label, m_col, m_group in metrics:
                    rho, pval = spearmanr(lc_sub[ev_col].values,
                                          lc_sub[m_col].values)
                    rows.append({
                        'subset': subset_name,
                        'event_source': ev_label,
                        'metric': m_label,
                        'group': m_group,
                        'land_cover': lc_name,
                        'lc_code': lc_code,
                        'n': len(lc_sub),
                        'spearman_rho': rho,
                        'p_value': pval,
                    })

    return pd.DataFrame(rows)


# ====================================================================
# MAIN
# ====================================================================

def main():
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir = out_dir / '_tmp'
    tmp_dir.mkdir(exist_ok=True)

    print("=" * 70)
    print("SCRIPT 1: Zonal Statistics & Spearman Correlations")
    print("=" * 70)

    # ---- 1. Hexagons ----
    print("\n[1] Loading hexagons...")
    hexgdf = gpd.read_file(HEX_PATH)
    print(f"    {len(hexgdf)} hexagons loaded.")
    if 'hex_id' not in hexgdf.columns:
        hexgdf['hex_id'] = hexgdf.index

    # ---- 2. War events ----
    print("\n[2] Counting war events per hexagon...")
    events = gpd.read_file(WAR_EVENTS_PATH)
    print(f"    {len(events)} events loaded.")
    if events.crs != hexgdf.crs:
        events = events.to_crs(hexgdf.crs)

    joined = gpd.sjoin(events, hexgdf[['hex_id', 'geometry']],
                       how='inner', predicate='within')

    layer_col = None
    for c in joined.columns:
        if c.lower() in ('layer', 'source', 'dataset'):
            layer_col = c; break
    if layer_col is None:
        for c in joined.columns:
            if 'layer' in c.lower() or 'source' in c.lower():
                layer_col = c; break

    if layer_col:
        print(f"    Layer column: '{layer_col}', "
              f"values: {joined[layer_col].unique()}")
        acled = (joined[joined[layer_col] == 'acled_simple']
                 .groupby('hex_id').size().rename('n_acled'))
        viina = (joined[joined[layer_col] == 'viina_simple']
                 .groupby('hex_id').size().rename('n_viina'))
    else:
        print("    WARNING: layer column not found.")
        acled = pd.Series(dtype=int, name='n_acled')
        viina = pd.Series(dtype=int, name='n_viina')

    combined = joined.groupby('hex_id').size().rename('n_combined')
    hexgdf = hexgdf.join(acled, on='hex_id').fillna({'n_acled': 0})
    hexgdf = hexgdf.join(viina, on='hex_id').fillna({'n_viina': 0})
    hexgdf = hexgdf.join(combined, on='hex_id').fillna({'n_combined': 0})
    for c in ['n_acled', 'n_viina', 'n_combined']:
        hexgdf[c] = hexgdf[c].astype(int)

    print(f"    Combined: {hexgdf['n_combined'].sum()}, "
          f"ACLED: {hexgdf['n_acled'].sum()}, "
          f"VIINA: {hexgdf['n_viina'].sum()}")

    # ---- 3. Z-score zonal stats ----
    print("\n[3] Z-score zonal statistics (band 7)...")
    z_tmp = str(tmp_dir / 'zscore_rf.tif')
    extract_band_tif(RF_PATH, ZSCORE_BAND, z_tmp)
    z_stats = zscore_zonal(hexgdf, z_tmp)
    for k, v in z_stats.items():
        hexgdf[k] = v
    print(f"    {(hexgdf['z_count'] > 0).sum()} hexagons with valid z-score data")

    # ---- 4. Confidence pixel counts ----
    print("\n[4] Confidence raster zonal pixel counts...")
    dmg_c, rew_c, val_c = confidence_zonal(hexgdf, CONF_PATH)
    hexgdf['conf_damage_px'] = dmg_c
    hexgdf['conf_rewild_px'] = rew_c
    hexgdf['conf_valid_px']  = val_c
    print(f"    Damage pixels total:    {hexgdf['conf_damage_px'].sum()}")
    print(f"    Rewilding pixels total: {hexgdf['conf_rewild_px'].sum()}")

    # ---- 5. Majority land cover ----
    print("\n[5] Majority land cover...")
    hexgdf['lc_code'] = majority_landcover(hexgdf, ESA_PATH)
    hexgdf['lc_name'] = hexgdf['lc_code'].map(ESA_LEGEND).fillna('No data')
    lc_dist = hexgdf['lc_name'].value_counts()
    print("    Distribution:")
    for lc, cnt in lc_dist.items():
        print(f"      {lc:25s}: {cnt:>6}")

    # ---- 6. Export hex dataset ----
    print("\n[6] Exporting hexagon dataset...")
    export_cols = [
        'hex_id', 'n_combined', 'n_acled', 'n_viina',
        'lc_code', 'lc_name',
        'z_median_damage', 'z_median_rewild',
        'z_max_damage', 'z_max_rewild', 'z_count',
        'conf_damage_px', 'conf_rewild_px', 'conf_valid_px',
    ]
    csv_path = out_dir / 'hexagon_stats.csv'
    hexgdf[export_cols].to_csv(csv_path, index=False, float_format='%.4f')
    print(f"    -> {csv_path}")
    gpkg_path = out_dir / 'hexagon_stats.gpkg'
    hexgdf[export_cols + ['geometry']].to_file(gpkg_path, driver='GPKG')
    print(f"    -> {gpkg_path}")

    # ---- 7. Spearman correlations ----
    print("\n[7] Computing Spearman correlations...")

    for ev_label, ev_col in [('Combined','n_combined'),
                              ('ACLED','n_acled'),
                              ('VIINA','n_viina')]:
        ss = make_subsets(hexgdf, ev_col)
        print(f"    {ev_label}: all_valid={ss['all_valid'].sum()}, "
              f"z_active={ss['z_active'].sum()}, "
              f"px_active={ss['px_active'].sum()}")

    event_cols = {
        'Combined': 'n_combined',
        'ACLED':    'n_acled',
        'VIINA':    'n_viina',
    }

    corr_df = compute_correlations(hexgdf, event_cols)

    corr_csv = out_dir / 'spearman_correlations.csv'
    corr_df.to_csv(corr_csv, index=False, float_format='%.6f')
    print(f"    -> {corr_csv}")

    # Summary
    for subset in ['all_valid', 'z_active', 'px_active']:
        sub = corr_df[(corr_df['land_cover'] == 'All') &
                      (corr_df['subset'] == subset)]
        if sub.empty:
            continue
        print(f"\n    ---- ALL LC, {subset.upper()} ----")
        for _, r in sub.iterrows():
            sig = ('***' if r['p_value'] < 0.001 else
                   '**'  if r['p_value'] < 0.01  else
                   '*'   if r['p_value'] < 0.05  else 'ns')
            print(f"    {r['event_source']:10s} x {r['metric']:18s}: "
                  f"rho={r['spearman_rho']:+.3f}  p={r['p_value']:.2e}  "
                  f"n={r['n']:>5}  {sig}")

    # ---- 8. Cleanup ----
    print("\n[8] Cleaning up...")
    for f in tmp_dir.glob('*.tif'):
        f.unlink()
    try:
        tmp_dir.rmdir()
    except OSError:
        pass

    print("\n" + "=" * 70)
    print("Done. Outputs in:", out_dir)
    print("=" * 70)


if __name__ == '__main__':
    main()

SCRIPT 1: Zonal Statistics & Spearman Correlations

[1] Loading hexagons...
    4740 hexagons loaded.

[2] Counting war events per hexagon...
    299901 events loaded.
    Layer column: 'layer', values: ['acled_simple' 'viina_simple']
    Combined: 299901, ACLED: 86525, VIINA: 213376

[3] Z-score zonal statistics (band 7)...
      500/4740
      1000/4740
      1500/4740
      2000/4740
      2500/4740
      3000/4740
      3500/4740
      4000/4740
      4500/4740
    2669 hexagons with valid z-score data

[4] Confidence raster zonal pixel counts...
      500/4740
      1000/4740
      1500/4740
      2000/4740
      2500/4740
      3000/4740
      3500/4740
      4000/4740
      4500/4740
    Damage pixels total:    11337
    Rewilding pixels total: 42947

[5] Majority land cover...
      500/4740
      1000/4740
      1500/4740
      2000/4740
      2500/4740
      3000/4740
      3500/4740
      4000/4740
      4500/4740
    Distribution:
      Cropland                 :   2098
   

In [ ]:
"""
Script 2 -- Plotting
====================
Reads hexagon_stats.csv and spearman_correlations.csv from Script 1.

Two figure sets:
  all_valid — heatmap + scatters using single all_valid mask (all 6 metrics)
  active    — heatmap merging z_active (4 z cols) + px_active (2 px cols);
              scatters: top row z_active mask, bottom row px_active mask

Style:
  - No plot titles
  - Colorbar: "Spearman's rho", fixed -1 to +1
  - Significance: *** / ** / * / ns in cell upper-right corner
  - Panel labels a-d in scatter upper-left
  - Font: e-Ukraine
  - Output: 900 dpi PNG (234 mm wide) + PDF
"""

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.font_manager as fm
from scipy.stats import spearmanr
from pathlib import Path

# ====================================================================
# CONFIGURATION
# ====================================================================

DATA_DIR = Path(r"C:\Users\karasoo1\Downloads\hex_analysis_mocked")
HEX_CSV  = DATA_DIR / 'hexagon_stats.csv'
CORR_CSV = DATA_DIR / 'spearman_correlations.csv'
OUT_DIR  = DATA_DIR / 'figures'

FONT_PATH = r"C:\Users\karasoo1\AppData\Local\Microsoft\Windows\Fonts\e-Ukraine-Regular.otf"

FIG_WIDTH_MM = 234
FIG_DPI      = 900
FIG_WIDTH_IN = FIG_WIDTH_MM / 25.4

MIN_N = 30

CLR_DAMAGE = '#d73027'
CLR_REWILD = '#4575b4'

# Maps internal metric names (matching CSV) to display labels shown on plots.
# Any name not listed here is displayed as-is.
METRIC_DISPLAY_NAMES = {
    'Med damage':     'Median damage',
    'Med warwilding': 'Median warwilding',
    'Max warwilding': 'Max warwilding',
    'Damage px':      'Damage pixels',
    'Warwilding px':  'Warwilding pixels',
}

def display_name(metric):
    """Return the display label for a metric, falling back to the raw name."""
    return METRIC_DISPLAY_NAMES.get(metric, metric)


# 2x2 scatter layout: top = z-score, bottom = pixel counts
# Tuple: (panel_label, internal_metric_name, column, color, type)
SCATTER_METRICS = [
    ('a', 'Max damage',     'z_max_damage',    CLR_DAMAGE,  'z'),
    ('b', 'Max warwilding', 'z_max_rewild',    CLR_REWILD,  'z'),
    ('c', 'Damage px',      'conf_damage_px',  CLR_DAMAGE,  'px'),
    ('d', 'Warwilding px',  'conf_rewild_px',  CLR_REWILD,  'px'),
]

# Internal names used to match rows in the correlations CSV
HEATMAP_COLS = [
    'Med damage', 'Med warwilding',
    'Max damage', 'Max warwilding',
    'Damage px', 'Warwilding px',
]

# Which heatmap columns come from which active subset
Z_COL_NAMES  = {'Med damage', 'Med warwilding', 'Max damage', 'Max warwilding'}
PX_COL_NAMES = {'Damage px', 'Warwilding px'}

ESA_LEGEND = {
    10: 'Tree cover', 20: 'Shrubland', 30: 'Grassland', 40: 'Cropland',
    50: 'Built-up', 60: 'Bare / sparse', 80: 'Water bodies',
    90: 'Herbaceous wetland',
}


# ====================================================================
# FONT SETUP
# ====================================================================

def setup_font():
    p = Path(FONT_PATH)
    font_family = None
    if p.exists():
        fm.fontManager.addfont(str(p))
        prop = fm.FontProperties(fname=str(p))
        font_family = prop.get_name()
        print(f"    Font registered: {font_family}")
    else:
        matches = [f for f in fm.fontManager.ttflist
                   if 'ukraine' in f.name.lower()]
        if matches:
            font_family = matches[0].name
            print(f"    Font found: {font_family}")
        else:
            print("    WARNING: e-Ukraine not found, using default.")
    rc = {
        'axes.titlesize': 10, 'axes.labelsize': 10,
        'xtick.labelsize': 10, 'ytick.labelsize': 10,
        'figure.dpi': 900,
    }
    if font_family:
        rc['font.family'] = font_family
    plt.rcParams.update(rc)


# ====================================================================
# HELPERS
# ====================================================================

def sig_label(pval):
    if pval < 0.001:  return '***'
    if pval < 0.01:   return '**'
    if pval < 0.05:   return '*'
    return 'ns'


def save_fig(fig, stem, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    png = out_dir / f'{stem}.png'
    pdf = out_dir / f'{stem}.pdf'
    fig.savefig(png, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"      -> {png.name}  /  {pdf.name}")


def safe_slug(name):
    return name.lower().replace(' ', '_').replace('/', '_')


def get_mask(hexdf, ev_col, subset, lc_code=None):
    """
    Replicate Script 1 subset logic exactly.
    """
    base = (hexdf['z_count'] > 0) & (hexdf['conf_valid_px'] > 0)
    if subset in ('z_active', 'px_active'):
        base = base & (hexdf[ev_col] > 0)
    if subset == 'px_active':
        base = base & ((hexdf['conf_damage_px'] > 0) |
                       (hexdf['conf_rewild_px'] > 0))
    if lc_code is not None:
        base = base & (hexdf['lc_code'] == lc_code)
    return base


# ====================================================================
# HEATMAP
# ====================================================================

def make_heatmap(corr_sub, col_labels, save_stem, out_dir):
    """
    corr_sub may contain results from different subsets (z_active + px_active
    merged for the 'active' heatmap). Each cell carries its own n.

    col_labels uses internal metric names (matching CSV values).
    Display labels are resolved via display_name() at render time.
    """
    if corr_sub.empty:
        return

    records = []
    for ev in ['ACLED', 'VIINA']:
        ev_sub = corr_sub[corr_sub['event_source'] == ev]
        if ev_sub.empty:
            continue
        all_r = ev_sub[ev_sub['land_cover'] == 'All']
        lc_r  = ev_sub[ev_sub['land_cover'] != 'All'].sort_values('lc_code')
        for _, r in all_r.iterrows():
            records.append({**r.to_dict(),
                            'row_label': r['event_source'] + ' -- All'})
        for _, r in lc_r.iterrows():
            records.append({**r.to_dict(),
                            'row_label': r['event_source'] + ' -- ' + r['land_cover']})

    if not records:
        return

    plot_df    = pd.DataFrame(records)
    row_labels = list(dict.fromkeys(plot_df['row_label']))

    rho_mat  = np.full((len(row_labels), len(col_labels)), np.nan)
    pval_mat = np.full((len(row_labels), len(col_labels)), np.nan)
    n_mat    = np.full((len(row_labels), len(col_labels)), np.nan)

    for _, r in plot_df.iterrows():
        ri = row_labels.index(r['row_label'])
        if r['metric'] in col_labels:
            ci = col_labels.index(r['metric'])
            rho_mat[ri, ci]  = r['spearman_rho']
            pval_mat[ri, ci] = r['p_value']
            n_mat[ri, ci]    = r['n']

    fig_h = max(3.5, 0.48 * len(row_labels) + 1.6)
    fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, fig_h))

    cmap = plt.cm.RdBu_r
    norm = mcolors.TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
    im = ax.imshow(rho_mat, cmap=cmap, norm=norm, aspect='auto')

    for i in range(len(row_labels)):
        for j in range(len(col_labels)):
            val = rho_mat[i, j]
            pv  = pval_mat[i, j]
            nv  = n_mat[i, j]

            if np.isnan(val):
                ax.text(j, i, '-', ha='center', va='center',
                        fontsize=10, color='grey')
                continue

            bg  = cmap(norm(val))
            lum = 0.299 * bg[0] + 0.587 * bg[1] + 0.114 * bg[2]
            tc  = 'white' if lum < 0.45 else 'black'

            ax.text(j, i, f'{val:+.2f}\n(n={int(nv)})',
                    ha='center', va='center',
                    fontsize=10, color=tc, fontweight='bold')

            sl = sig_label(pv)
            sig_c = tc if sl != 'ns' else (
                'white' if lum < 0.45 else '#999999')
            ax.text(j + 0.44, i - 0.38, sl,
                    ha='right', va='top', fontsize=8, color=sig_c,
                    fontstyle='italic' if sl == 'ns' else 'normal')

    for i, label in enumerate(row_labels):
        if '-- All' in label and i > 0:
            ax.axhline(i - 0.5, color='black', lw=1.0)

    ax.set_xticks(range(len(col_labels)))
    # Apply display names then wrap on spaces for compact tick labels
    split_labels = [display_name(lbl).replace(' ', '\n') for lbl in col_labels]
    ax.set_xticklabels(split_labels, fontsize=10, fontweight='bold')
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10, fontweight='bold')

    cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label("Spearman's rho", fontsize=12)
    cbar.set_ticks([-1.0, -0.5, 0.0, 0.5, 1.0])

    fig.tight_layout()
    save_fig(fig, save_stem, out_dir)


# ====================================================================
# 2x2 SCATTER
# ====================================================================

def make_scatter_2x2(hexdf, ev_col, ev_label, mode,
                     lc_code=None, save_stem=None, out_dir=None):
    """
    mode = 'all_valid' -> all 4 panels use all_valid mask
    mode = 'active'    -> top row (z) uses z_active, bottom row (px) uses px_active
    """
    fig, axes = plt.subplots(2, 2,
                             figsize=(FIG_WIDTH_IN, FIG_WIDTH_IN * 0.82))

    for idx, (panel_lbl, metric_name, m_col, color, mtype) in enumerate(SCATTER_METRICS):
        ax = axes[idx // 2, idx % 2]

        # Pick correct mask
        if mode == 'all_valid':
            subset_key = 'all_valid'
        else:
            subset_key = 'z_active' if mtype == 'z' else 'px_active'

        mask = get_mask(hexdf, ev_col, subset_key, lc_code)
        sub = hexdf[mask]

        ax.text(0.03, 0.97, panel_lbl, transform=ax.transAxes,
                fontsize=11, fontweight='bold', va='top', ha='left')

        # Use display name for the y-axis label
        ylabel = display_name(metric_name)

        if len(sub) < 5:
            ax.text(0.5, 0.5, 'n < 5', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color='grey')
            ax.set_xlabel(f'log10({ev_label} + 1)', fontsize=8)
            ax.set_ylabel(ylabel, fontsize=8)
            continue

        x = np.log10(sub[ev_col].values + 1)
        y = sub[m_col].values
        rho, pval = spearmanr(sub[ev_col].values, y)
        sl = sig_label(pval)

        ax.scatter(x, y, c=color, alpha=0.25, s=10,
                   edgecolors='none', rasterized=True)

        if len(sub) > 10:
            coeffs = np.polyfit(x, y, 1)
            xf = np.linspace(x.min(), x.max(), 100)
            ax.plot(xf, np.polyval(coeffs, xf), 'k-', lw=1.2, alpha=0.7)

        ax.set_xlabel(f'log10({ev_label} + 1)', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)

        p_str = 'p < 0.001' if pval < 0.001 else f'p = {pval:.3f}'
        txt = f'rho = {rho:+.3f} {sl}\n{p_str}\nn = {len(sub)}'
        ax.text(0.97, 0.97, txt, transform=ax.transAxes, fontsize=10,
                va='top', ha='right', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.3', fc='white',
                          ec='grey', alpha=0.85))

    fig.tight_layout(pad=1.5)
    save_fig(fig, save_stem, out_dir)


# ====================================================================
# MAIN
# ====================================================================

def main():
    print("=" * 70)
    print("SCRIPT 2: Plotting")
    print("=" * 70)

    print("\n[0] Font setup...")
    setup_font()

    print("\n[1] Loading data...")
    corr_df = pd.read_csv(CORR_CSV)
    hexdf   = pd.read_csv(HEX_CSV)
    print(f"    Correlations: {len(corr_df)} rows")
    print(f"    Hexagons:     {len(hexdf)} rows")

    # Combined event source excluded; ACLED and VIINA only
    event_sources = [
        ('ACLED', 'n_acled'),
        ('VIINA', 'n_viina'),
    ]

    # ------------------------------------------------------------------
    # 2. HEATMAPS
    # ------------------------------------------------------------------
    print("\n[2] Heatmaps...")

    # 2a. all_valid — all 6 metrics from one subset
    av_corr = corr_df[corr_df['subset'] == 'all_valid']
    av_dir  = OUT_DIR / 'all_valid'
    print("  all_valid:")
    make_heatmap(av_corr, HEATMAP_COLS, 'heatmap_overview', av_dir)

    # 2b. active — merge z_active (4 z cols) + px_active (2 px cols)
    z_act  = corr_df[(corr_df['subset'] == 'z_active') &
                     (corr_df['metric'].isin(Z_COL_NAMES))]
    px_act = corr_df[(corr_df['subset'] == 'px_active') &
                     (corr_df['metric'].isin(PX_COL_NAMES))]
    active_corr = pd.concat([z_act, px_act], ignore_index=True)
    act_dir = OUT_DIR / 'active'
    print("  active (z_active + px_active):")
    make_heatmap(active_corr, HEATMAP_COLS, 'heatmap_overview', act_dir)

    # ------------------------------------------------------------------
    # 3. SCATTERPLOTS
    # ------------------------------------------------------------------
    print("\n[3] Scatterplots...")

    for mode, mode_dir in [('all_valid', av_dir), ('active', act_dir)]:
        print(f"\n  --- {mode} ---")

        # Determine gatekeeper subset for per-LC n check
        gate_subset = 'all_valid' if mode == 'all_valid' else 'z_active'

        for ev_label, ev_col in event_sources:
            ev_tag = ev_label.lower()

            # All land cover
            print(f"    {ev_label} / All...")
            make_scatter_2x2(
                hexdf, ev_col, ev_label, mode,
                save_stem=f'scatter_{ev_tag}_all', out_dir=mode_dir)

            # Per land cover
            for lc_code, lc_name in sorted(ESA_LEGEND.items()):
                n_lc = get_mask(hexdf, ev_col, gate_subset,
                                lc_code=lc_code).sum()
                if n_lc < MIN_N:
                    continue
                slug = safe_slug(lc_name)
                print(f"    {ev_label} / {lc_name} (n={n_lc})...")
                make_scatter_2x2(
                    hexdf, ev_col, ev_label, mode,
                    lc_code=lc_code,
                    save_stem=f'scatter_{ev_tag}_{slug}',
                    out_dir=mode_dir)

    # ------------------------------------------------------------------
    print("\n" + "=" * 70)
    print(f"Done. Figures in: {OUT_DIR}")
    print(f"  {FIG_DPI} dpi, {FIG_WIDTH_MM} mm wide")
    print("=" * 70)


if __name__ == '__main__':
    main()

SCRIPT 2: Plotting

[0] Font setup...
    Font registered: e-Ukraine

[1] Loading data...
    Correlations: 138 rows
    Hexagons:     4740 rows

[2] Heatmaps...
  all_valid:
      -> heatmap_overview.png  /  heatmap_overview.pdf
  active (z_active + px_active):
      -> heatmap_overview.png  /  heatmap_overview.pdf

[3] Scatterplots...

  --- all_valid ---
    ACLED / All...
      -> scatter_acled_all.png  /  scatter_acled_all.pdf
    ACLED / Tree cover (n=327)...
      -> scatter_acled_tree_cover.png  /  scatter_acled_tree_cover.pdf
    ACLED / Grassland (n=217)...
      -> scatter_acled_grassland.png  /  scatter_acled_grassland.pdf
    ACLED / Cropland (n=2098)...
      -> scatter_acled_cropland.png  /  scatter_acled_cropland.pdf
    VIINA / All...
      -> scatter_viina_all.png  /  scatter_viina_all.pdf
    VIINA / Tree cover (n=327)...
      -> scatter_viina_tree_cover.png  /  scatter_viina_tree_cover.pdf
    VIINA / Grassland (n=217)...
      -> scatter_viina_grassland.png  /  sc